In [1]:
import nest_asyncio
import os
import pickle
from llama_index.core import VectorStoreIndex

In [2]:
with open("embeddings.pkl", "rb") as f:
    embeds = pickle.load(f)

In [2]:
from pinecone.grpc import PineconeGRPC
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext

In [23]:
from dotenv import load_dotenv
from pinecone import ServerlessSpec
load_dotenv()
api = os.getenv("PINECONE_API_KEY")

pine = PineconeGRPC(api_key=api)


In [ ]:
pine.create_index("embeddings", dimension=1024, metric='cosine', spec=ServerlessSpec(cloud="aws",region="us-east-1"))

In [23]:
with open("extracts.pkl", "rb") as f:
    nodes = pickle.load(f)

In [4]:
index = pine.Index("embeddings")

In [5]:
vectorstore = PineconeVectorStore(pinecone_index=index)

In [27]:
vectors=[]
for i, embed in enumerate(embeds):
    nodal = {
        "id" : f"{i}",
        "values": embed,
        "metadata": { "text": nodes[i].text, **nodes[i].metadata}
        
    }
    vectors.append(nodal)


In [32]:
for i in range(0,len(vectors), 100):
    batch = vectors[i:i+100]
    if i == 9500:
        batch = vectors[i:]
    index.upsert(vectors=batch)
    print("upserted ",i)

upserted  0
upserted  100
upserted  200
upserted  300
upserted  400
upserted  500
upserted  600
upserted  700
upserted  800
upserted  900
upserted  1000
upserted  1100
upserted  1200
upserted  1300
upserted  1400
upserted  1500
upserted  1600
upserted  1700
upserted  1800
upserted  1900
upserted  2000
upserted  2100
upserted  2200
upserted  2300
upserted  2400
upserted  2500
upserted  2600
upserted  2700
upserted  2800
upserted  2900
upserted  3000
upserted  3100
upserted  3200
upserted  3300
upserted  3400
upserted  3500
upserted  3600
upserted  3700
upserted  3800
upserted  3900
upserted  4000
upserted  4100
upserted  4200
upserted  4300
upserted  4400
upserted  4500
upserted  4600
upserted  4700
upserted  4800
upserted  4900
upserted  5000
upserted  5100
upserted  5200
upserted  5300
upserted  5400
upserted  5500
upserted  5600
upserted  5700
upserted  5800
upserted  5900
upserted  6000
upserted  6100
upserted  6200
upserted  6300
upserted  6400
upserted  6500
upserted  6600
upserte

In [18]:
from llama_index.llms.groq import Groq
load_dotenv()

True

In [19]:
api = os.getenv("GROQ_API_KEY")
llm = Groq(model="openai/gpt-oss-120b", api=api)

In [20]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer

In [24]:
from llama_index.core import Settings
Settings.llm = llm

from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
import nest_asyncio

nest_asyncio.apply()
embed_model = HuggingFaceInferenceAPIEmbedding(
    model_name="BAAI/bge-m3",
    token=os.getenv("HF_API_KEY"),
)
Settings.embed_model=embed_model

vectorindex = VectorStoreIndex.from_vector_store(vector_store=vectorstore, llm=llm)

retriver = VectorIndexRetriever(index=vectorindex,similarity_top_k=10 )
synth = get_response_synthesizer()
query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth)
response = query_engine.query("Outline some technological techniques of conservation ")

In [22]:
print(response)

**Techniques commonly used in conservation planning and practice**

- **Goal‑setting and target definition**  
  - Clearly identify what species, habitats, or ecosystem services need protection.  
  - Translate those targets into measurable conservation goals (e.g., “protect 30 % of the remaining native vegetation”).

- **Information gathering and gap analysis**  
  - Compile existing ecological, genetic, and land‑use data.  
  - Pinpoint missing data or methodological shortcomings and decide whether to collect new information or adjust analytical approaches.

- **Biodiversity assessment and prioritization**  
  - Evaluate areas for their biodiversity values, species richness, and ecosystem‑service provision.  
  - Use spatial analyses (habitat‑suitability models, connectivity assessments) to rank sites for protection, restoration, or mitigation.

- **Strategic analysis and scenario testing**  
  - Run data‑driven models to explore how different actions meet the set goals and identify 

In [ ]:
query = input("Enter your query: ")

from llama_index.core import RichPromptTemplate

prompt_instructions = f'You are a part of a conservation research RAG assistant - improve the prompt for the LLM to yield better results. The user has entered the prompt below:\n {query} \n The prompt could have limitations such as being too specific or too generalised for a corpus detailing technical insights into species conservation whilst not providing contraints to the LLM. \n Your job is to enhance the query so that RAG retrieval returns interesting points of discussion all backed by citations, recognition of uncertainty of information and a breakdown of information reasoning. \n in Ensure the new prompt is between 50-60 words max'


prompt_template = Rich

"""
schema for design: 

user enters: conservation techniques for this and that

rephrase with prompt: (query expansion)
provide a sophisticated analysis of the point of disucssion. Follow the schema provided to generate a thoughtful report of the question based on the corpus. Ensure that all claims are referenced with page number and file name. Draw conclusions based on the information from the corpus. if you cannot find any reasonable link with the corpus, make it clear that there are gaps in place. 

schema: 
{
    'Introduction' : pass
    'Key insights and strategies':
    'Confidence score /10':
    'List of statistics':
    'References':
}

new prompt is taken and entered into the vector db. Results returned. 

Query stratgies - reranking, 

"""